# Train PINN v2 on Colab (T4)

Trains the lens-model PINN (`deeplense/models/pinn_lens.py`) with the Poisson loss on (λ = 0.1)
and the physics-off ablation (λ = 0).

**Before running:** *Runtime → Change runtime type → T4 GPU*.

### Surviving disconnects
Free Colab disconnects after ~90 min without interaction, after at most ~12 h, or when the daily
GPU quota runs out. You can't fully prevent that, so this notebook makes it harmless instead:

- Results and a full checkpoint (`last.pt`: model, optimizer, scheduler, history) are written to
  **Google Drive after every epoch**.
- `--resume` continues from that checkpoint. After a disconnect: reconnect, run all cells again,
  and training picks up at the next epoch. Finished runs are skipped.
- Mixed precision (`--amp`) and batch 64 make each run short, so most runs finish in one session.

To reduce the chance of disconnects: keep this tab open and in front, keep your computer awake and
online, and check in occasionally. Closing the tab is the most common cause.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Drive, code, dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/DeepLense'
OUT = '/content/drive/MyDrive/DeepLense_results'      # results + checkpoints persist here
ZIP = '/content/drive/MyDrive/dataset1.zip'           # the dataset zip already on your Drive
DATA = '/content/dataset'
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
!test -d {REPO} && git -C {REPO} pull -q || git clone -q https://github.com/AWASTHI-ANANT/DeepLense.git {REPO}
!git -C {REPO} log -1 --oneline
!pip -q install timm

## 2. Data → local disk
Reading 37,500 small files straight from the Drive mount is very slow, so copy the zip once to the
Colab VM's local disk and unzip there (~1–2 min). Re-run after a disconnect: the VM disk is wiped.

In [ ]:
import os, glob
if not os.path.isdir(f'{DATA}/train'):
    !cp "{ZIP}" /content/data.zip && unzip -q -o /content/data.zip -d /content/ && rm /content/data.zip
for split in ['train', 'val']:
    print(split, {c: len(glob.glob(f'{DATA}/{split}/{c}/*.npy')) for c in ['no', 'sphere', 'vort']})

## 3. Train
Each command prints one line per epoch (time, losses, `theta_e_px`, val AUC), writes a log to Drive,
and resumes automatically if a checkpoint exists. A finished run (with `metrics.json`) is skipped.

In [ ]:
import subprocess, sys

def train(lam):
    run = f'pinn2_lambda{lam:g}'
    if os.path.exists(f'{OUT}/{run}/metrics.json'):
        print(f'{run}: already finished, skipping'); return
    cmd = [sys.executable, '-u', 'scripts/train.py', '--model', 'pinn2', '--lambda-poisson', str(lam),
           '--data-root', DATA, '--out-dir', OUT, '--batch-size', '64', '--num-workers', '2',
           '--amp', '--resume']
    with open(f'{OUT}/{run}.log', 'a') as log:
        proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:            # stream each epoch line to the cell and the Drive log
            if 'HF_TOKEN' not in line:
                print(line, end=''); log.write(line); log.flush()
    if proc.wait() != 0:
        raise RuntimeError(f'{run} failed (exit {proc.returncode}); see the output above')

train(0.1)    # Poisson loss on

In [ ]:
train(0)      # ablation: same model, Poisson loss off

## 4. Results

In [ ]:
import json
for run in ['pinn2_lambda0.1', 'pinn2_lambda0']:
    f = f'{OUT}/{run}/metrics.json'
    if os.path.exists(f):
        t = json.load(open(f))['test']
        print(f"{run:18s} test acc {t['accuracy']:.4f}  macro AUC {t['macro_auc']:.4f}  ",
              {k.split()[0]: round(v, 4) for k, v in t['auc_per_class'].items()})

Then download the two run folders (`metrics.json`, `history.json`, `best.pt`,
`test_predictions.npz`) from `MyDrive/DeepLense_results/` into the repo's `results/` folder.
`last.pt` is only needed for resuming and can be left behind.